In [ ]:
import numpy as np
from scipy.stats import norm

def climate_vasicek(pd0, q, â, y, rho=0.15):
    CDT = norm.ppf(pd0)
    IRD = np.sqrt(1 - rho)
    
    z1 = (CDT + np.sqrt(rho) * y) / IRD
    z2 = (CDT + np.sqrt(rho) * y + â) / IRD
    
    return (1 - q) * norm.cdf(z1) + q * norm.cdf(z2)

countries = {
    'Spain': {
        'pd0': 0.051, 
        'q': 0.30, 
        'â': 0.1542, 
        'y': 1.397154502, 
        'rho': 0.15, 
        'p_obs': 0.052
    },
    'Turkey': {
        'pd0': 0.0252, 
        'q': 0.29, 
        'â': 0.2062, 
        'y': 0.2457693747, 
        'rho': 0.15, 
        'p_obs': 0.0385
    },
    'Serbia': {
        'pd0': 0.032, 
        'q': 0.26, 
        'â': 0.0813, 
        'y': 0.000472687, 
        'rho': 0.15, 
        'p_obs': 0.049
    }   
}

errors_climate = [climate_pd - p['p_obs'] for p in countries.values()]
errors_macro   = [macro_pd - p['p_obs']   for p in countries.values()]

mae_climate = np.mean(np.abs(errors_climate))
rmse_climate = np.sqrt(np.mean(np.square(errors_climate)))

print(f"Climate Model MAE:  {mae_climate:.4%}")
print(f"Climate Model RMSE: {rmse_climate:.4%}")

for country_name, p in countries.items():

    climate_pd = climate_vasicek(
        pd0=p['pd0'], q=p['q'], â=p['â'], y=p['y'], rho=p['rho']
    )
    
  
    macro_pd = climate_vasicek(
        pd0=p['pd0'], q=p['q'], â=0.0, y=p['y'], rho=p['rho']
    )
    
    climate_penalty = climate_pd - macro_pd
    model_residual  = climate_pd - p['p_obs']
    
    print(f"--- {country_name} ---")
    print(f"Macro Baseline PD (â=0):   {macro_pd:.4%}")
    print(f"Climate Model PD (â>0):    {climate_pd:.4%}")
    print(f"Marginal Climate Penalty:  {climate_penalty:+.4%}")
    print(f"Observed Target NPL:       {p['p_obs']:.4%}")
    print(f"Model Residual / Bias:     {model_residual:+.4%}\n")

Climate Model MAE:  2.2890%
Climate Model RMSE: 2.3611%
--- Spain ---
Macro Baseline PD (â=0):   11.7665%
Climate Model PD (â>0):    12.8562%
Marginal Climate Penalty:  +1.0896%
Observed Target NPL:       5.2000%
Model Residual / Bias:     +7.6562%

--- Turkey ---
Macro Baseline PD (â=0):   2.1747%
Climate Model PD (â>0):    2.5969%
Marginal Climate Penalty:  +0.4222%
Observed Target NPL:       3.8500%
Model Residual / Bias:     -1.2531%

--- Serbia ---
Macro Baseline PD (â=0):   2.2281%
Climate Model PD (â>0):    2.3610%
Marginal Climate Penalty:  +0.1329%
Observed Target NPL:       4.9000%
Model Residual / Bias:     -2.5390%

